In [2]:
import os, time, requests, shutil
import pandas as pd
import numpy as np
import calendar
from google.colab import drive

drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_TUGAS = "/content/drive/MyDrive/BigData/Praktikum3_Tugas"

for d in (DIR_MENTAH, DIR_KURASI, DIR_TUGAS):
    os.makedirs(d, exist_ok=True)

# 1. Fungsi Utilitas & Akuisisi Dasar
def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk: f.write(chunk)
            os.replace(sementara, tujuan)
            return tujuan
        except Exception as e:
            time.sleep(jeda_awal * (2**i))
    raise RuntimeError(f"Gagal unduh: {url}")

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": mulai,
              "end_date": selesai, "hourly": "temperature_2m,precipitation",
              "timezone": "America/New_York"}
    r = requests.get(url, params=params)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df.rename(columns={"time": "jam_mulai", "temperature_2m": "suhu_c", "precipitation": "hujan_mm"})

# SUMBER KEEMPAT: API Hari Libur Nasional US (Nager.Date)
def ambil_libur(tahun=2023):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{tahun}/US"
    r = requests.get(url)
    r.raise_for_status()
    df = pd.DataFrame(r.json())[["date", "name"]]
    df["date"] = pd.to_datetime(df["date"]).dt.date
    return df.rename(columns={"date": "tanggal", "name": "nama_libur"})

# 2. Persiapan Data Referensi
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")
unduh_aman(URL_ZONA, PATH_ZONA)
zona = pd.read_csv(PATH_ZONA)

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis", "Sengketa", "Tidak diketahui", "Perjalanan batal"]
})

libur = ambil_libur(2023)

# 3. Fungsi Pemrofilan & Kualitas Data
def buat_laporan_kualitas(df, bulan_tahun):
    awal = pd.Timestamp(f"{bulan_tahun}-01")
    akhir = awal + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)
    zona_sah = set(zona["LocationID"])

    aturan = {
        "kelengkapan: passenger_count ada": df["passenger_count"].notna(),
        "keunikan: baris tidak duplikat": ~df.duplicated(),
        "validitas: jarak 0-100 mil": df["trip_distance"].between(0.01, 100),
        "validitas: total_amount > 0": df["total_amount"] > 0,
        "validitas: PULocationID dikenal": df["PULocationID"].isin(zona_sah),
        "konsistensi: durasi 1-180 menit": df["durasi_menit"].between(1, 180),
        "konsistensi: total >= fare": df["total_amount"] >= df["fare_amount"],
        "ketepatan waktu dalam bulan target": df["tpep_pickup_datetime"].between(awal, akhir),
    }

    laporan = pd.DataFrame([
        {"aturan": nama, "gagal": int((~mask).sum()), "persen_gagal": round(100 * (~mask).mean(), 3)}
        for nama, mask in aturan.items()
    ])
    laporan["bulan"] = bulan_tahun
    return laporan

# 4. Pipeline Utama
KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
         "trip_distance", "PULocationID", "DOLocationID", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "total_amount"]

def proses_bulan(bulan_tahun):
    print(f"\nMemproses data bulan: {bulan_tahun}")
    url_trip = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{bulan_tahun}.parquet"
    path_trip = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan_tahun}.parquet")
    unduh_aman(url_trip, path_trip)

    df = pd.read_parquet(path_trip, columns=KOLOM)
    df["durasi_menit"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

    laporan_kualitas = buat_laporan_kualitas(df, bulan_tahun)

    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(df.groupby("jam")["passenger_count"].transform("median"))
    df = df.drop_duplicates(subset=KUNCI)

    df = df[df["trip_distance"].between(0.01, 100) &
            df["durasi_menit"].between(1, 180) &
            (df["total_amount"] > 0)]

    df = (df.merge(zona[["LocationID", "Borough"]].rename(columns={"Borough": "borough_naik"}),
                   left_on="PULocationID", right_on="LocationID", how="left")
            .drop(columns="LocationID")
            .merge(tarif_referensi, on="payment_type", how="left"))

    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")

    awal_str = f"{bulan_tahun}-01"
    akhir_str = (pd.Timestamp(awal_str) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    cuaca_bulan = ambil_cuaca(awal_str, akhir_str)

    df = df.merge(cuaca_bulan, on="jam_mulai", how="left")

    # Integrasi Sumber Keempat
    df["tanggal"] = df["tpep_pickup_datetime"].dt.date
    df = df.merge(libur, on="tanggal", how="left")
    df["nama_libur"] = df["nama_libur"].fillna("Bukan Libur")

    # Penulisan Parquet Berpartisi berdasarkan bulan dan borough
    df["bulan_partisi"] = bulan_tahun
    out_dir = os.path.join(DIR_KURASI, "trips_gabungan")
    df.to_parquet(out_dir, partition_cols=["bulan_partisi", "borough_naik"], index=False)

    print(f"Selesai memproses {bulan_tahun}. Total baris akhir: {len(df):,}")
    return laporan_kualitas

# 5. Eksekusi Dua Bulan dan Laporan Komparatif
laporan_jan = proses_bulan("2023-01")
laporan_jul = proses_bulan("2023-07")

laporan_komparatif = laporan_jan.merge(laporan_jul, on="aturan", suffixes=("_Jan", "_Jul"))
laporan_komparatif.to_csv(os.path.join(DIR_TUGAS, "laporan_kualitas_komparatif.csv"), index=False)

print("\n=== Laporan Kualitas Komparatif ===")
print(laporan_komparatif[["aturan", "persen_gagal_Jan", "persen_gagal_Jul"]])

Mounted at /content/drive

Memproses data bulan: 2023-01
Selesai memproses 2023-01. Total baris akhir: 2,986,910

Memproses data bulan: 2023-07
Selesai memproses 2023-07. Total baris akhir: 2,817,657

=== Laporan Kualitas Komparatif ===
                               aturan  persen_gagal_Jan  persen_gagal_Jul
0    kelengkapan: passenger_count ada             2.339             2.927
1      keunikan: baris tidak duplikat             0.000             0.000
2          validitas: jarak 0-100 mil             1.498             1.717
3         validitas: total_amount > 0             0.840             1.075
4     validitas: PULocationID dikenal             0.000             0.000
5     konsistensi: durasi 1-180 menit             1.186             1.311
6          konsistensi: total >= fare             0.816             1.053
7  ketepatan waktu dalam bulan target             0.002             0.002
